# Module 7.4 — Self-RAG

Self-RAG uses **reflection tokens** to decide:
- Whether to retrieve at all (`[Retrieve]` / `[No Retrieve]`)
- Whether retrieved docs are relevant (`[Relevant]` / `[Irrelevant]`)
- Whether the generated answer is supported (`[Supported]` / `[Contradicts]`)
- Whether the answer is useful (`[Utility: 1–5]`)

In [ ]:
from langchain_openai import ChatOpenAI, OpenAIEmbeddings
from langchain_community.vectorstores import Chroma
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain.schema import Document

llm = ChatOpenAI(model="gpt-4o-mini", temperature=0)

docs = [
    Document(page_content="The speed of light in vacuum is approximately 299,792,458 metres per second."),
    Document(page_content="Einstein's theory of relativity shows mass and energy are equivalent: E=mc²."),
    Document(page_content="Photons are massless particles that always travel at the speed of light."),
]
embeddings = OpenAIEmbeddings(model="text-embedding-3-small")
vs         = Chroma.from_documents(docs, embeddings, collection_name="selfrag_demo")

# ── Reflection token: should we retrieve? ────────────────────────────────────
def needs_retrieval(query: str) -> bool:
    prompt = ChatPromptTemplate.from_template("""
Does answering this question require looking up external knowledge?
Answer only 'yes' or 'no'.
Question: {query}
""")
    result = (prompt | llm | StrOutputParser()).invoke({"query": query})
    return "yes" in result.lower()

# ── Reflection token: is doc relevant? ───────────────────────────────────────
def is_relevant(query: str, doc_content: str) -> bool:
    prompt = ChatPromptTemplate.from_template("""
Is this document relevant to the question? Answer only 'yes' or 'no'.
Question: {query}
Document: {doc}
""")
    result = (prompt | llm | StrOutputParser()).invoke({"query": query, "doc": doc_content})
    return "yes" in result.lower()

# ── Reflection token: is answer grounded? ────────────────────────────────────
def is_grounded(answer: str, context: str) -> bool:
    prompt = ChatPromptTemplate.from_template("""
Is this answer fully supported by the context? Answer 'yes' or 'no'.
Context: {context}
Answer: {answer}
""")
    result = (prompt | llm | StrOutputParser()).invoke({"answer": answer, "context": context})
    return "yes" in result.lower()

# ── Self-RAG pipeline ─────────────────────────────────────────────────────────
def self_rag(query: str) -> dict:
    print(f"\nQuery: {query}")

    # Step 1: retrieval decision
    retrieve = needs_retrieval(query)
    print(f"  [Retrieve?] {retrieve}")

    context = ""
    if retrieve:
        candidates = vs.similarity_search(query, k=3)
        relevant   = [d for d in candidates if is_relevant(query, d.page_content)]
        print(f"  [Relevant docs] {len(relevant)}/{len(candidates)}")
        context = "\n".join(d.page_content for d in relevant)

    # Step 2: generate
    gen_prompt = ChatPromptTemplate.from_template("""
Answer the question{ctx_note}.
{context}
Question: {query}
""")
    answer = (gen_prompt | llm | StrOutputParser()).invoke({
        "query": query, "context": f"Context:\n{context}" if context else "",
        "ctx_note": " using the context below" if context else ""
    })

    # Step 3: grounding check
    grounded = is_grounded(answer, context) if context else True
    print(f"  [Grounded?] {grounded}")
    print(f"  Answer: {answer.strip()[:200]}")
    return {"answer": answer, "grounded": grounded, "retrieved": retrieve}


self_rag("What is the speed of light?")
self_rag("What is your favourite colour?")
